In [1]:

import os
from dotenv import load_dotenv
import requests
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

TELEGRAM_TOKEN = os.getenv('TELEGRAM_TOKEN')
TELEGRAM_CHAT_ID = os.getenv('TELEGRAM_CHAT_ID')


if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

client = OpenAI(api_key=api_key)


API key found and looks good so far!


In [3]:
system_prompt = """
You are a professional Recruitment Scout. Your task is to extract job vacancies for "Manual QA", "QA Automation", or "SDET".

STRICT FORMATTING RULES:

1. NUMBERING: Every item MUST start with a number (e.g., 1., 2., 3.).
2. LINK INTEGRATION: Do NOT write the word "Link" at the end. Instead, make the Job Title a clickable Markdown link.
3. TEMPLATE: [**Senior QA Automation Engineer**](https://example.com/job1) | QA Automation | 📍 Berlin | 📅 2 days ago
4. CONTENT: Extract only QA-related roles.
5. LANGUAGE: Use English for all output.
"""

In [5]:
user_prompt_prefix = """
Analyze the website content below and list the QA vacancies found. 
Follow the example format exactly:
1. [**Senior QA Automation Engineer**](https://example.com/job1) | QA Automation | 📍 Berlin | 📅 2 days ago

**Website Content to Analyze:**
---
"""

In [6]:
def fetch_website_contents(url):
    """Downloads website content using Jina Reader to bypass bot protection."""
    full_url = f"https://r.jina.ai/{url}"
    print(f"📡 Fetching data for: {url}")
    try:
        response = requests.get(full_url, timeout=20)
        return response.text if response.status_code == 200 else f"Error: Status {response.status_code}"
    except Exception as e:
        return f"Connection error: {str(e)}"



In [7]:

def messages_for(website_text):
    """Prepares the message list for the OpenAI API."""
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_text}
    ]

In [8]:

def summarize(url):
    """Scrapes the URL and gets the AI analysis."""
    content = fetch_website_contents(url)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for(content)
    )
    return response.choices[0].message.content

In [9]:

def send_telegram_message(text):
    """Sends the job list to your Telegram bot."""
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": text,
        "parse_mode": "Markdown"
    }
    try:
        res = requests.post(url, json=payload)
        return res.status_code == 200
    except:
        return False

def get_new_vacancies_only(current_summary, url_id):
    """Prevents duplicate alerts by comparing current results with history."""
    # We use a unique filename for each search URL to avoid mixing them up
    history_file = f"history_{url_id}.txt"
    
    old_summary = ""
    if os.path.exists(history_file):
        with open(history_file, "r", encoding="utf-8") as f:
            old_summary = f.read()

    if current_summary.strip() != old_summary.strip() and "No new" not in current_summary:
        with open(history_file, "w", encoding="utf-8") as f:
            f.write(current_summary)
        return current_summary
    return None

In [10]:
def run_automation(url, url_name, test_mode=False):
    """Main execution flow for a single search URL."""
    print(f"🔍 Checking updates for: {url_name}...")
    summary = summarize(url)
    
    if test_mode:
        print(f"🛠 TEST MODE: Results for {url_name}:")
        display(Markdown(summary))
    else:
        new_jobs = get_new_vacancies_only(summary, url_name)
        if new_jobs:
            print(f"✨ New jobs found for {url_name}! Sending to Telegram...")
            send_telegram_message(f"🚀 **New Jobs: {url_name}**\n\n{new_jobs}")
            display(Markdown(new_jobs))
        else:
            print(f"😴 No new updates for {url_name}.")

# 4. EXECUTION (TRACK MULTIPLE SEARCHES)

if __name__ == "__main__":
    # Define your job search targets here
    job_targets = {
        "Stepstone_QA_Automation": "https://www.stepstone.de/jobs/qa-automation",
        "Stepstone_Manual_QA": "https://www.stepstone.de/jobs/manual-qa",
        # You can add more URLs here
    }

 

In [11]:
   # Set to True if you just want to see results in Cursor without Telegram/History
TESTING = False 

for name, link in job_targets.items():
    run_automation(link, name, test_mode=TESTING)
print("-" * 30)

🔍 Checking updates for: Stepstone_QA_Automation...
📡 Fetching data for: https://www.stepstone.de/jobs/qa-automation
✨ New jobs found for Stepstone_QA_Automation! Sending to Telegram...


1. [**QA Engineer**](https://www.stepstone.de/jobs/qa-engineer) | Manual QA | 📍 Langen | 📅 14 hours ago  
2. [**Team Lead QA**](https://www.stepstone.de/jobs/team-lead-qa) | QA Automation | 📍 Berlin | 📅 3 days ago  
3. [**Junior QA Test Automation**](https://www.stepstone.de/jobs/junior-qa-test-automation) | QA Automation | 📍 Berlin | 📅 1 week ago  
4. [**Working Student QA – Performance Testing**](https://www.stepstone.de/jobs/working-student-qa-performance-testing) | Manual QA | 📍 Berlin | 📅 1 day ago  
5. [**(Senior) Test Automation Engineer**](https://www.stepstone.de/jobs/senior-test-automation-engineer) | QA Automation | 📍 Munich | 📅 2 days ago  
6. [**Test Automation Engineer**](https://www.stepstone.de/jobs/test-automation-engineer) | QA Automation | 📍 Stuttgart | 📅 1 week ago  
7. [**Software Tester / QA Engineer**](https://www.stepstone.de/jobs/software-tester-qa-engineer) | Manual QA | 📍 Hamburg | 📅 5 days ago  
8. [**QA Automation Engineer**](https://www.stepstone.de/jobs/qa-automation-engineer) | QA Automation | 📍 Frankfurt | 📅 2 weeks ago  

🔍 Checking updates for: Stepstone_Manual_QA...
📡 Fetching data for: https://www.stepstone.de/jobs/manual-qa
✨ New jobs found for Stepstone_Manual_QA! Sending to Telegram...


1. [**Working Student – QA Test Engineer (Embedded & Cloud) (d/f/m) - Digital Transformation**](https://www.stepstone.de/stellenangebote--Working-Student-QA-Test-Engineer-Embedded-Cloud-d-f-m-Digital-Transformation-Duesseldorf-TK-Elevator-GmbH--14004643-inline.html) | Manual QA | 📍 Düsseldorf | 📅 6 days ago  
2. [**Quality Assurance Engineer (m/f/d)**](https://www.stepstone.de/stellenangebote--Quality-Assurance-Engineer-m-f-d-Leer-Forterro-Deutschland-GmbH--14061514-inline.html) | QA Automation | 📍 Leer | 📅 6 days ago  
3. [**QA Tester - Manual Testing & User Experience (w/m/d)**](https://www.stepstone.de/stellenangebote--QA-Tester-Manual-Testing-User-Experience-w-m-d-Berlin-eClear-AG--12875515-inline.html) | Manual QA | 📍 Berlin | 📅 1 week ago  
4. [**QA Engineer - Core Database (remote)**](https://www.stepstone.de/stellenangebote--QA-Engineer-Core-Database-remote-Germany-remote-Clickhouse-GmbH--13906167-inline.html) | QA Automation | 📍 Germany (remote) | 📅 2 days ago  
5. [**Student Worker**](https://www.stepstone.de/stellenangebote--Student-Worker-Dusseldorf-Germany-BlackBerry-Deutschland-GmbH--14045480-inline.html) | Manual QA | 📍 Düsseldorf, Germany | 📅 1 week ago  
6. [**QA CSV Specialist (m/w/d)**](https://www.stepstone.de/stellenangebote--QA-CSV-Specialist-m-w-d-Hechingen-Bentley-InnoMed-GmbH--14092416-inline.html) | Manual QA | 📍 Hechingen | 📅 8 hours ago  
7. [**Software Quality Assurance / QA Engineer (m/w/d)**](https://www.stepstone.de/stellenangebote--Software-Quality-Assurance-QA-Engineer-m-w-d-Langen-Hessen-Advancis-Software-Services-GmbH--14091326-inline.html) | QA Automation | 📍 Langen (Hessen) | 📅 14 hours ago  
8. [**QA Deviation & Complaint Manager (m/w/d)**](https://www.stepstone.de/stellenangebote--QA-Deviation-Complaint-Manager-m-w-d-Ennigerloh-Rottendorf-Pharma-GmbH--14070027-inline.html) | Manual QA | 📍 Ennigerloh | 📅 4 days ago  
9. [**CRM Salesforce/ Application Testing & QA Specialist (m/w/d)**](https://www.stepstone.de/stellenangebote--CRM-Salesforce-Application-Testing-QA-Specialist-m-w-d-Kiel-CITTI-Handelsgesellschaft-mbH-Co-KG--14060906-inline.html) | QA Automation | 📍 Kiel | 📅 6 days ago  
10. [**Expert QA Operations (m/w/d) - Operational Technology (OT) Oversight**](https://www.stepstone.de/stellenangebote--Expert-QA-Operations-m-w-d-Operational-Technology-OT-Oversight-Pfaffenhofen-an-der-Ilm-Daiichi-Sankyo-Europe-GmbH--14015050-inline.html) | QA Automation | 📍 Pfaffenhofen an der Ilm | 📅 4 days ago  
11. [**IT-Qualitätssicherung / QA Engineer (m/w/d) - Schwerpunkt Webanwendungen**](https://www.stepstone.de/stellenangebote--IT-Qualitaetssicherung-QA-Engineer-m-w-d-Schwerpunkt-Webanwendungen-Hannover-fb-research-GmbH--14000451-inline.html) | QA Automation | 📍 Hannover | 📅 1 week ago  
12. [**QA Automation Engineer (gn)**](https://www.stepstone.de/stellenangebote--QA-Automation-Engineer-gn-Berlin-OPTIMAL-SYSTEMS-GmbH--14050335-inline.html) | QA Automation | 📍 Berlin | 📅 1 week ago  
13. [**QA-Engineer / Softwaretester (m/w/d)**](https://www.stepstone.de/stellenangebote--QA-Engineer-Softwaretester-m-w-d-Salem-am-Bodensee-INNOSYSTEC-GmbH--14035004-inline.html) | Manual QA | 📍 Salem am Bodensee | 📅 1 week ago  

------------------------------
